# Stage 1 — Non-Instruction (Continued Pretraining) Fine-Tuning

**Hummingbird Assistant LLM**

**Environment:** Lightning.ai Studio, T4 GPU (16GB VRAM). Persistent storage is Lightning.ai's Teamspace path, which survives Studio restarts.

**Base model:** `unsloth/Llama-3.2-3B` (base, bnb-4bit prequantized)

Storage model: **no local-only save**. Every artifact is saved to the Lightning.ai Teamspace path (persistent, equivalent to what Drive was in the Colab version) and then uploaded to a **single Hugging Face Hub repo**, organized into per-stage subfolders (`stage1-adapter/`, `stage1-merged/`, and later `stage2-adapter/`, `stage2-merged/`, `stage3-adapter/`, `stage3-merged/`).

This notebook:
1. Loads the pre-built domain corpus (`data/non_instruction_data.txt`)
2. Loads `unsloth/Llama-3.2-3B` in 4-bit and attaches a fresh LoRA adapter
3. Runs continued pretraining with `SFTTrainer`/`SFTConfig` (`packing=True`, raw text, no instruction format)
4. Saves the adapter to the Teamspace path, merges into a full 16-bit model, saves that too, then uploads both folders to the shared HF Hub repo under `stage1-adapter/` and `stage1-merged/`
5. Reloads the merged model from the Teamspace path and runs the same 10 questions used in `base_model_evaluation.ipynb`

## 1. Install libraries

In [1]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets
!pip -q install huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.9 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.6.7 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.


## 2. Imports

In [2]:
import os
import gc
import time
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
from datasets import Dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

from huggingface_hub import HfApi, notebook_login

assert torch.cuda.is_available(), "GPU not found. On Lightning.ai: select a GPU-backed Studio (e.g. T4) before running."
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## 3. Config
`HF_REPO` is a single shared repo used by **all three stages** — each stage writes to its own subfolder within it. `LIGHTNING_ROOT` is Lightning.ai's persistent Teamspace path.

In [3]:
# ---- File paths ----
non_instruction_data_path = "data/non_instruction_data.txt"

# ---- Model ----
BASE_MODEL_NAME = "unsloth/Llama-3.2-3B"  # base (non-instruct) model, upgraded from Llama-3.2-1B
MAX_SEQ_LENGTH = 512
SEED = 42

# ---- QLoRA hyperparameters (consistent across all three training stages) ----
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

BATCH_SIZE = 5
GRAD_ACCUM_STEPS = 4
WARMUP_STEPS = 10
LOGGING_STEPS = 5
NUM_TRAIN_EPOCHS = 20

STAGE1_LR = 2e-4

# ---- Persistent storage (Lightning.ai Teamspace path; the ONLY local-disk write) ----
LIGHTNING_ROOT = "/teamspace/studios/this_studio/hummingbird-assistant-llm"
STAGE1_ADAPTER_DIR = f"{LIGHTNING_ROOT}/stage1-adapter"
STAGE1_MERGED_DIR  = f"{LIGHTNING_ROOT}/stage1-merged"

# ---- CONFIGURE THIS with your own HF username ----
# Single shared repo for ALL stages -- each stage uploads into its own subfolder.
HF_REPO = "Snow79703/Hummingbird-assistant-llm"

## 4. Set up persistent storage and log in to Hugging Face Hub
Lightning.ai Teamspace paths are persistent by default across Studio restarts, so we just ensure the directories exist.

In [5]:
os.makedirs(STAGE1_ADAPTER_DIR, exist_ok=True)
os.makedirs(STAGE1_MERGED_DIR, exist_ok=True)
print(f"Persistent storage ready at: {LIGHTNING_ROOT}")

Persistent storage ready at: /teamspace/studios/this_studio/hummingbird-assistant-llm


In [7]:
notebook_login()  # paste a Hugging Face write token when prompted

hf_api = HfApi()
hf_api.create_repo(repo_id=HF_REPO, exist_ok=True, repo_type="model")
print(f"HF Hub repo ready: https://huggingface.co/{HF_REPO}")

HF Hub repo ready: https://huggingface.co/Snow79703/Hummingbird-assistant-llm


## 5. Helper functions

In [8]:
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def generate_completion(model, tokenizer, prompt: str, max_new_tokens: int = 80):
    """Plain raw-text completion -- no instruction template. Appropriate for
    Stage 1, since the model has not been instruction-tuned yet."""
    FastLanguageModel.for_inference(model)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    Used identically at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads Stage 1's merged Teamspace folder
    - Stage 3 loads Stage 2's merged Teamspace folder
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str,
                            hf_repo: str, stage_prefix: str, stage_name: str):
    """
    Saves the LoRA adapter and the merged 16-bit model to the Lightning.ai
    Teamspace path (the only local-disk write, used as persistent storage),
    then uploads both folders into the SHARED hf_repo under
    f"{stage_prefix}-adapter/" and f"{stage_prefix}-merged/" using
    upload_folder(path_in_repo=...), since push_to_hub() cannot target a subfolder.
    """
    print(f"\nSaving {stage_name} adapter to {adapter_dir} ...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to: {adapter_dir}")

    print(f"\nMerging {stage_name} adapter with base model, saving merged model to {merged_dir} ...")
    FastLanguageModel.for_training(model)
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    print(f"{stage_name} merged model saved to: {merged_dir}")

    print(f"\nUploading {stage_name} adapter to https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-adapter ...")
    hf_api.upload_folder(
        repo_id=hf_repo,
        folder_path=adapter_dir,
        path_in_repo=f"{stage_prefix}-adapter",
        repo_type="model",
    )

    print(f"Uploading {stage_name} merged model to https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-merged ...")
    hf_api.upload_folder(
        repo_id=hf_repo,
        folder_path=merged_dir,
        path_in_repo=f"{stage_prefix}-merged",
        repo_type="model",
    )

    print(f"\n{stage_name} complete. Teamspace: {adapter_dir}, {merged_dir}")
    print(f"HF Hub: https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-adapter")
    print(f"HF Hub: https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-merged")

## 6. Stage 1 data: load the pre-built domain corpus
Loads the already-generated, copyright-safe `data/non_instruction_data.txt` (paraphrased content, not extracted paper text). If it isn't present in the Studio's working directory, this raises a clear error rather than silently continuing.

In [9]:
def build_text_dataset(path: str) -> Dataset:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. On Lightning.ai, add this file to the Studio first -- "
            f"e.g. clone the repo (`git clone ...`), drag-and-drop it via the Studio "
            f"file browser, or `scp` it in -- then re-run this cell."
        )

    with open(path, "r", encoding="utf-8") as f:
        full_text = f.read()

    paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]

    if len(paragraphs) == 0:
        raise ValueError("No usable paragraphs found in the corpus file.")

    print("Paragraph records:", len(paragraphs))
    print("\nSample paragraph:\n", paragraphs[0][:400])

    return Dataset.from_dict({"text": paragraphs})


stage1_dataset = build_text_dataset(non_instruction_data_path)

Paragraph records: 52

Sample paragraph:
 Hummingbirds (family Trochilidae) are distinguished by sustained hovering flight, the smallest body sizes among birds, and metabolic rates that rank among the highest of any vertebrate. These traits make them valuable subjects for population health research, yet their solitary and highly mobile lifestyles have historically limited data collection. With over 360 recognized species concentrated in t


## 7. Stage 1: Non-instruction continued pretraining
Trains with `packing=True` since these are short, unrelated paragraphs rather than a single continuous document — packing combines multiple paragraphs into full-length training sequences for efficiency.

In [10]:
print("\n==============================")
print("STAGE 1: NON-INSTRUCTION TRAINING")
print("==============================")

clear_gpu_memory()
stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir="/tmp/stage1_logs",  # ephemeral trainer logs only, not a deliverable

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,

    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION TRAINING")


STAGE 1: NON-INSTRUCTION TRAINING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Unsloth: Tokenizing ["text"] (num_proc=7):   0%|          | 0/52 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=7):   0%|          | 0/52 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14 | Num Epochs = 20 | Total steps = 20
O^O/ \_/ \    Batch size per device = 5 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (5 x 4 x 1) = 20
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.949300
10,2.566900
15,1.837700
20,1.336200



STAGE 1 - NON-INSTRUCTION TRAINING RESULTS
Train time/sec: 134.42
Peak allocated VRAM/GB: 3.732
Peak reserved VRAM/GB: 4.52


TrainOutput(global_step=20, training_loss=2.1725301265716555, metrics={'train_runtime': 131.774, 'train_samples_per_second': 2.125, 'train_steps_per_second': 0.152, 'total_flos': 1716754103992320.0, 'train_loss': 2.1725301265716555, 'epoch': 20.0})

## 8. Save adapter, merge, and upload (Teamspace + shared HF Hub repo, subfoldered)

In [11]:
save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    hf_repo=HF_REPO,
    stage_prefix="stage1",
    stage_name="Stage 1",
)

del stage1_trainer
del stage1_model
clear_gpu_memory()


Saving Stage 1 adapter to /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-adapter ...
Stage 1 adapter saved to: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-adapter

Merging Stage 1 adapter with base model, saving merged model to /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-merged ...
Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading Stage 1 merged model to https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage1-merged ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Stage 1 complete. Teamspace: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-adapter, /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-merged
HF Hub: https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage1-adapter
HF Hub: https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage1-merged


## 9. Inference demo (on the freshly reloaded MERGED model from Teamspace storage)

Loads the just-saved merged model fresh from the Teamspace path (not the in-memory adapter model, which has already been deleted above) and runs the **same 10 questions** asked of the raw base model in `base_model_evaluation.ipynb`. Questions are fed as **plain raw text, no instruction wrapper** — Stage 1 has not been instruction-tuned yet, so this still measures raw completion quality, not Q&A behavior. Also prints a paste-ready Markdown table for `reports/sft_model_comparison.md`.

In [12]:
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(inference_model)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm

In [13]:
eval_questions = [
    "What is your knowledge cutoff date?",
    "What is a hummingbird's heart rate, and how does it change during flight?",
    "How fast can hummingbirds fly, including during a courtship dive?",
    "What is torpor in hummingbirds and why do they use it?",
    "How far do rufous hummingbirds migrate each year?",
    "What is the typical wingbeat frequency of a hovering hummingbird?",
    "How many eggs does a female hummingbird lay per clutch?",
    "What is the smallest hummingbird species in the world?",
    "What are the main threats to hummingbird populations today?",
    "Why can hummingbirds hover while most other birds cannot?",
    "What adaptations allow hummingbirds to consume nectar and process it so efficiently?" # added this one

]

stage1_results = []
for i, q in enumerate(eval_questions, start=1):
    completion = generate_completion(inference_model, inference_tokenizer, q, max_new_tokens=150)
    stage1_results.append({"id": i, "question": q, "stage1_answer": completion})
    print(f"Q{i}: {q}")
    print(f"A{i}: {completion}")
    print("-" * 80)

Q1: What is your knowledge cutoff date?
A1: For the purposes of determining whether a disease or condition qualifies as a listed vector-borne illness, the Agency considers any disease identified by the U.S. Centers for Disease Control and Prevention (CDC) or World Health Organization (WHO) through surveillance or outbreak investigations to be appropriately documented beyond that date. The most recent updated list was published in September 2017; subsequent updates will be posted on the website as new information becomes available.
How does the agency determine what constitutes "documentation"? For diseases or conditions listed in the Vector-Borne Illnesses fact sheet, documentation includes confirmed or probable laboratory-confirmed cases reported to CDC or WHO since 2000, unless otherwise noted. In some instances, a case count rather than individual case counts alone has been
--------------------------------------------------------------------------------
Q2: What is a hummingbird's h

### Markdown table (paste-ready for `reports/sft_model_comparison.md`)

In [14]:
def to_markdown_table(results, answer_key, header):
    lines = [f"| # | Question | {header} |", "|---|---|---|"]
    for r in results:
        q = r["question"].replace("|", "\\|")
        a = r[answer_key].replace("\n", " ").replace("|", "\\|")
        lines.append(f"| {r['id']} | {q} | {a} |")
    return "\n".join(lines)


print(to_markdown_table(stage1_results, "stage1_answer", "Stage 1 (Non-Instruction) Answer"))

| # | Question | Stage 1 (Non-Instruction) Answer |
|---|---|---|
| 1 | What is your knowledge cutoff date? | For the purposes of determining whether a disease or condition qualifies as a listed vector-borne illness, the Agency considers any disease identified by the U.S. Centers for Disease Control and Prevention (CDC) or World Health Organization (WHO) through surveillance or outbreak investigations to be appropriately documented beyond that date. The most recent updated list was published in September 2017; subsequent updates will be posted on the website as new information becomes available. How does the agency determine what constitutes "documentation"? For diseases or conditions listed in the Vector-Borne Illnesses fact sheet, documentation includes confirmed or probable laboratory-confirmed cases reported to CDC or WHO since 2000, unless otherwise noted. In some instances, a case count rather than individual case counts alone has been |
| 2 | What is a hummingbird's heart rate, 

## Done

Stage 1 outputs:
- **Teamspace (persistent):** `STAGE1_ADAPTER_DIR`, `STAGE1_MERGED_DIR`
- **HF Hub (shared repo, subfoldered):** `https://huggingface.co/{HF_REPO}/tree/main/stage1-adapter`, `https://huggingface.co/{HF_REPO}/tree/main/stage1-merged`

Stage 2 (`instruction_finetuning.ipynb`) reuses the same helper functions, the same `unsloth/Llama-3.2-3B` lineage, and loads Stage 1's merged Teamspace folder (or downloads `stage1-merged/` from the shared HF repo) as its starting checkpoint, then uploads its own results under `stage2-adapter/` and `stage2-merged/` in the same shared repo.

In [15]:
print("Stage 1 adapter (Teamspace):", STAGE1_ADAPTER_DIR)
print("Stage 1 merged model (Teamspace):", STAGE1_MERGED_DIR)
print("Stage 1 adapter (HF Hub):", f"https://huggingface.co/{HF_REPO}/tree/main/stage1-adapter")
print("Stage 1 merged model (HF Hub):", f"https://huggingface.co/{HF_REPO}/tree/main/stage1-merged")

Stage 1 adapter (Teamspace): /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-adapter
Stage 1 merged model (Teamspace): /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-merged
Stage 1 adapter (HF Hub): https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage1-adapter
Stage 1 merged model (HF Hub): https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage1-merged
